# Davinci Notebook 16 — End-to-End Integration Test (Tiny Configuration)

This is a gate before full pretraining. It tests a small causal model, synthetic or tiny token corpus, optimizer step, validation, checkpoint save/reload, and continuation. It does not verify the full 500M architecture or dataset pipeline unless those are explicitly substituted.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, tempfile, pathlib, math
from torch.utils.data import Dataset,DataLoader
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; print('device',DEVICE)


In [ ]:
class TinyCausalLM(nn.Module):
 def __init__(self,vocab=128,d=64): super().__init__(); self.e=nn.Embedding(vocab,d); self.norm=nn.LayerNorm(d); self.h=nn.Linear(d,vocab); self.vocab=vocab
 def forward(self,ids): return self.h(self.norm(self.e(ids)))
class TinyTokenDataset(Dataset):
 def __init__(self,seqs): self.seqs=seqs
 def __len__(self): return len(self.seqs)
 def __getitem__(self,i): return torch.tensor(self.seqs[i],dtype=torch.long)
def lm_loss(logits,ids): return F.cross_entropy(logits[:,:-1].float().reshape(-1,logits.size(-1)),ids[:,1:].reshape(-1))


## 1. Run multiple optimizer steps and validation

In [ ]:
torch.manual_seed(42)
data = torch.randint(0, 128, (64, 64))
train_loader = DataLoader(TinyTokenDataset(data[:48]), batch_size=4, shuffle=True)
val_loader = DataLoader(TinyTokenDataset(data[48:]), batch_size=4)

model = TinyCausalLM().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = []
for step, batch in enumerate(train_loader):
    if step >= 5:
        break
    batch = batch.to(DEVICE)
    optimizer.zero_grad(set_to_none=True)
    logits = model(batch)
    loss = lm_loss(logits, batch)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    history.append(float(loss.detach()))

@torch.no_grad()
def val_loss(model, loader):
    model.eval()
    values = []
    for batch in loader:
        batch = batch.to(DEVICE)
        values.append(float(lm_loss(model(batch), batch)))
    return sum(values) / max(1, len(values))

validation = val_loss(model, val_loader)
print({"train_losses": history, "validation_loss": validation})
assert len(history) == 5
assert all(math.isfinite(value) for value in history)
assert math.isfinite(validation)


## 2. Save, reload, and resume checkpoint

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
 path=pathlib.Path(tmp)/'tiny.pt'; torch.save({'model':m.state_dict(),'optimizer':opt.state_dict(),'step':5},path)
 new=TinyCausalLM().to(DEVICE); newopt=torch.optim.AdamW(new.parameters(),lr=1e-3); ck=torch.load(path,map_location=DEVICE,weights_only=True); new.load_state_dict(ck['model']); newopt.load_state_dict(ck['optimizer'])
 test=data[:2].to(DEVICE); m.eval(); new.eval()
 with torch.no_grad(): err=(m(test)-new(test)).abs().max().item()
 print({'checkpoint_reload_error':err,'step':ck['step']}); assert err==0.0


## 3. Gate interpretation

Passing proves only that this isolated miniature training/checkpoint workflow functions. It does **not** establish correctness of the Davinci decoder, attention caching, corpus preprocessing, tokenizer, SFT, tool behavior, or long context. Replace this toy model with the exported Davinci module and a tiny real corpus in a later integration test.